# AdamOps Framework Test
Testing all main components using the New York Restaurants dataset.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import adamops
from adamops.data.loaders import load_csv
from adamops.data.preprocessors import preprocess
from adamops.data.splitters import split_train_test
from adamops.models.modelops import train, train_regression
from adamops.models.ensembles import create_voting_ensemble
from adamops.models.automl import run as automl_run
from adamops.evaluation.metrics import evaluate_model, compare_results

print('All imports successful.')

In [ ]:
# Load and preprocess data
data_path = '../test-data/newyork-rest.csv'
df = load_csv(data_path)

# Drop non-numeric and irrelevant columns for price prediction
df_clean = df.drop(columns=['Case', 'Restaurant'])
df_preprocessed = preprocess(df_clean, missing_strategy="mean", outlier_method="iqr", remove_duplicates=True)
df_preprocessed.head()

In [ ]:
# Split the data into features (X) and target (y)
X = df_preprocessed.drop(columns=['Price'])
y = df_preprocessed['Price']

X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

In [ ]:
# 1. Base Model Training
rf_model = train_regression(X_train, y_train, algorithm='random_forest')
ridge_model = train_regression(X_train, y_train, algorithm='ridge')

print("Fitted Random Forest and Ridge models.")

In [ ]:
# 2. Ensemble Modeling
ensemble_model = create_voting_ensemble(algorithms=['random_forest', 'ridge'], task='regression')
ensemble_model.fit(X_train, y_train)

print("Fitted Voting Ensemble.")

In [ ]:
# 3. AutoML
automl_result = automl_run(
    X_train, y_train, 
    task='regression', 
    backend='local',
    algorithms=['random_forest', 'ridge'],
    n_trials=10, 
    cv=3
)
best_automl_model = automl_result.best_model

print(f"AutoML Best Score (Negative MSE): {automl_result.best_score:.4f}")
print(f"AutoML Best Params: {automl_result.best_params}")

In [ ]:
# 4. Evaluation & Comparison
rf_metrics = evaluate_model(rf_model, X_test, y_test, task='regression')
ridge_metrics = evaluate_model(ridge_model, X_test, y_test, task='regression')
ensemble_metrics = evaluate_model(ensemble_model, X_test, y_test, task='regression')
automl_metrics = evaluate_model(best_automl_model, X_test, y_test, task='regression')

comparison_df = compare_results(
    [rf_metrics, ridge_metrics, ensemble_metrics, automl_metrics],
    names=['Random Forest', 'Ridge', 'Voting Ensemble', 'AutoML']
)
comparison_df